La elaboración y entrega de los reportes regulatorios, es una obligación fundamental en el cumplimiento de las instituciones financieras, estos son enviados (principalmente) a organismos que regulación el sistema financiero, como puede ser al banco central para que este se cerciore del estado operativo y financiero de la institución, pero sobre todo de su liquidez. Este notebook replica la solución a un requirimiento que un regulador estableció a las instituciones financieras en México, el mandato fue que se le reportaran el top 3 de operaciones por día, además del total por todas las operaciones por día (incluyendo el top 3) esto con un identificador en especial (el número 10).

In [1]:
import pandas as pd
import numpy as np

In [ ]:
# Cargar datos de liquidez operativa
data = pd.read_excel(r'data\3 mas altos por dia.xlsx', sheet_name='operativo')
data.head()

,Fecha,Moneda,Importe
0,2024-01-06,MXP,387706
1,2024-01-02,USD,2582
2,2024-01-04,USD,4870
3,2024-01-04,USD,3274
4,2024-01-02,MXP,159765


In [ ]:
# cargar los tipos de cambio, en este ejemplo solo se maneja dólar, en un escenario real hay por lo menos 5 divisas
tc = pd.read_excel(r'data\3 mas altos por dia.xlsx', sheet_name='TC')
tc.sample(5)

,Fecha,Moneda,TC
1,2024-01-04,USD,20.67
2,2024-01-01,USD,20.30
3,2024-01-06,USD,20.51
5,2024-01-03,USD,20.45
4,2024-01-05,USD,20.45


In [4]:
# 1. limpieza inicial (opcional)
data.columns = data.columns.str.strip()
tc.columns = tc.columns.str.strip()

# normalizar moneda
data['Moneda'] = data['Moneda'].str.strip().str.upper()
tc['Moneda'] = tc['Moneda'].str.strip().str.upper()

# normalizar fecha
data['Fecha'] = pd.to_datetime(data['Fecha'], dayfirst=True, errors='coerce')
tc['Fecha'] = pd.to_datetime(tc['Fecha'], dayfirst=True, errors='coerce')

In [ ]:
# 2. unión de ambas tablas con base en la fecha

# primero, para utilizar el método merge_asof() es necesario ordenar los datos, en este caso por fecha
data = data.sort_values(by='Fecha')
tc = tc.sort_values(by='Fecha')

df = pd.merge_asof(
    data, 
    tc[['Fecha','TC']], 
    on='Fecha', 
    direction='backward',
    suffixes=['_d','_tc'])

# asignar TC = 1.0 a MXP
df.loc[df['Moneda'] == 'MXP', 'TC'] = 1.0

# diagnóstico de filas sin TC (solo deberían ser monedas realmente faltantes)
faltantes = df[df['TC'].isna()]
if not faltantes.empty:
    print("Filas sin tipo de cambio (no son MXP):")
    print(faltantes)
    print("\nRevisa estas monedas/fechas en la tabla tc.\n")

In [ ]:
# 3. convertir el importe a MXP
df['Importe_MXP'] = np.where(
    df['Moneda'] == 'MXP',  # cuando Moneda sea MXP
    df['Importe'],          # no convertir MXP, dejar Importe
    df['Importe'] * df['TC']  # si Moneda no es MXP, convertir otras monedas
)

df.head()

,Fecha,Moneda,Importe,TC,Importe_MXP
0,2024-01-01,USD,4202,20.3,85300.6
1,2024-01-01,USD,6845,20.3,138953.5
2,2024-01-01,USD,6830,20.3,138649.0
3,2024-01-01,USD,1520,20.3,30856.0
4,2024-01-01,MXP,249520,1.0,249520.0


In [ ]:
# 4. identificar las tres operaciones con mayor importe por día
top_values = (
    df.sort_values(['Fecha', 'Importe_MXP'], ascending=[True, False])
      .groupby('Fecha')
      .head(3)
)

top_values.head()

,Fecha,Moneda,Importe,TC,Importe_MXP
7,2024-01-01,MXP,453835,1.00,453835.00
5,2024-01-01,MXP,382461,1.00,382461.00
4,2024-01-01,MXP,249520,1.00,249520.00
15,2024-01-02,MXP,248174,1.00,248174.00
13,2024-01-02,USD,8252,20.38,168175.76


In [ ]:
# 5. incluir el total de todas las operaciones por día, incluyendo las tres principales
total_importe_mxp = (
    df.groupby('Fecha', as_index=False)['Importe_MXP']
      .sum()
)
total_importe_mxp.head()

,Fecha,Importe_MXP
0,2024-01-01,2.31e+06
1,2024-01-02,7.70e+05
2,2024-01-03,3.69e+05
3,2024-01-04,1.18e+06
4,2024-01-05,1.07e+06


In [ ]:
# agregar columnas faltantes, este formato lo estableció el regulador y así debía presentarse
total_importe_mxp['Moneda'] = '10'
total_importe_mxp['TC'] = '' # np.nan
total_importe_mxp['Importe'] = '' # np.nan
total_importe_mxp.head()

,Fecha,Importe_MXP,Moneda,TC,Importe
0,2024-01-01,2.31e+06,10,,
1,2024-01-02,7.70e+05,10,,
2,2024-01-03,3.69e+05,10,,
3,2024-01-04,1.18e+06,10,,
4,2024-01-05,1.07e+06,10,,


In [ ]:
# 6. combinar y ordenar
combined = pd.concat([top_values, total_importe_mxp], ignore_index=True)

# marcar totales con un identificador, en este ejemplo con un 10  
combined['es_total'] = combined['Moneda'].eq('10')

# orden final:
# 1) Fecha ascendente
# 2) Detalles primero, totales al final
# 3) Detalles ordenados por Importe_MXP descendente
combined = combined.sort_values(
    ['Fecha', 'es_total', 'Importe_MXP'],
    ascending=[True, True, False]
).reset_index(drop=True)

# asegurar que la fecha se presente sin hora
combined['Fecha'] = combined['Fecha'].dt.date

# quitar serie auxiliar para el ordenado final
combined = combined.drop(columns='es_total')

In [11]:
# 7. resultado final
combined = combined[['Fecha','Moneda','TC','Importe','Importe_MXP']] # solo mostrar las columnas relevantes
combined.style.format(precision=2, thousands=',')

,Fecha,Moneda,TC,Importe,Importe_MXP
0,2024-01-01,MXP,1.00,"453,835","453,835.00"
1,2024-01-01,MXP,1.00,"382,461","382,461.00"
2,2024-01-01,MXP,1.00,"249,520","249,520.00"
3,2024-01-01,10,,,"2,312,889.90"
4,2024-01-02,MXP,1.00,"248,174","248,174.00"
5,2024-01-02,USD,20.38,"8,252","168,175.76"
6,2024-01-02,MXP,1.00,"159,765","159,765.00"
7,2024-01-02,10,,,"770,438.06"
8,2024-01-03,USD,20.45,"7,210","147,444.50"
9,2024-01-03,MXP,1.00,"138,413","138,413.00"


### Conclusión

Pareciera trivial la entrega de información operativa/financiera a un regulador, por experiencia no lo es en absoluto, debe haber mucho rigor en la información que se presente, aquí es donde las áreas de cumplimiento, control interno y auditoría deben de apoyar a la Organización, pero sobre todo auditoría debe aportar **valor**. Excel es la herramienta que por excelencia se maneja de forma extendida en las instituciones, sin embargo está demostrado que tiende a aceptar todo tipo de errores, incluídos algunos involuntarios por _tipeo_, en mi opinión el script, aunque simple, resume perfecto todo un pieline ETL real en python: extracción (E) de la información operativa, transformación (T) de los datos, incluyendo la limpieza y creación de nueva información a partir de los datos cargados (_feature engineering_) y su carga (L) en un ambiente de producción, ya sea para su validación final o directamente para ser guardado en un repositorio (Azure, AWS, servidor, etc.)    